In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0


In [2]:
# A simple tensor
t = torch.tensor([1.0, 2.0, 3.0])
print("Tensor:", t)
print("Shape:", t.shape)
print("Type:", t.dtype)

# A 2D tensor (like a tiny grayscale image)
img = torch.zeros(4, 4)  # 4x4 grid of zeros
print("\n4x4 image tensor:")
print(img)
print("Shape:", img.shape)

# Simulate a batch of 32 grayscale 28x28 images
batch = torch.zeros(32, 1, 28, 28)
print("Batch shape:", batch.shape)
print("Number of images:", batch.shape[0])
print("Image dimensions:", batch.shape[2], "x", batch.shape[3])

Tensor: tensor([1., 2., 3.])
Shape: torch.Size([3])
Type: torch.float32

4x4 image tensor:
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
Shape: torch.Size([4, 4])
Batch shape: torch.Size([32, 1, 28, 28])
Number of images: 32
Image dimensions: 28 x 28


In [3]:
# Transform converts images to tensors and normalizes pixel values
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download and load MNIST
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print("Training images:", len(train_data))
print("Test images:", len(test_data))

# Look at one image
image, label = train_data[0]
print("\nOne image shape:", image.shape)
print("Label:", label)

def filter_binary(dataset, class_a=0, class_b=1):
    # Get indices where label is either class_a or class_b
    indices = [i for i, (_, label) in enumerate(dataset) 
               if label == class_a or label == class_b]
    
    # Subset the dataset to only those indices
    subset = torch.utils.data.Subset(dataset, indices)
    return subset

train_binary = filter_binary(train_data)
test_binary = filter_binary(test_data)

train_loader = DataLoader(train_binary, batch_size=32, shuffle=True)
test_loader = DataLoader(test_binary, batch_size=32, shuffle=False)

print("Training batches:", len(train_loader))
print("Test batches:", len(test_loader))

print("Training images (0s and 1s only):", len(train_binary))
print("Test images (0s and 1s only):", len(test_binary))

# Verify it worked
image, label = train_binary[0]
print("First label:", label)

Training images: 60000
Test images: 10000

One image shape: torch.Size([1, 28, 28])
Label: 5
Training batches: 396
Test batches: 67
Training images (0s and 1s only): 12665
Test images (0s and 1s only): 2115
First label: 0


In [4]:
class ClassicalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 64)
        self.fc2 = nn.Linear(64, 2)  # Binary classification
        self.reul = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.reul(self.fc1(x))
        x = self.fc2(x)
        return x
    
model = ClassicalNet()
print(model)

ClassicalNet(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
  (reul): ReLU()
)
